In [13]:
import pandas as pd
import psycopg2
from textblob import TextBlob

# Load CSV
df = pd.read_csv("cleaned_reviews.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower()

print(df.columns)

# Rename columns for easier handling
df.rename(columns={
    'review text': 'review_text',
    'rating (1–5)': 'rating',
    'review date': 'review_date',
    'bank / app name': 'bank_name'
}, inplace=True)

# Convert review date
df['review_date'] = pd.to_datetime(df['review_date']).dt.date

# Sentiment analysis
def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        label = 'positive'
    elif polarity < 0:
        label = 'negative'
    else:
        label = 'neutral'

    return pd.Series([label, polarity])

df[['sentiment_label', 'sentiment_score']] = df['review_text'].apply(get_sentiment)

# Simple theme extraction
def identify_theme(text):
    text = str(text).lower()

    if 'login' in text:
        return 'Login Issues'
    elif 'transfer' in text:
        return 'Money Transfer'
    elif 'slow' in text:
        return 'Performance'
    else:
        return 'General'

df['identified_theme'] = df['review_text'].apply(identify_theme)

# PostgreSQL connection
conn = psycopg2.connect(
    host="localhost",
    database="Bank_reviews",
    user="postgres",
    password="MihiretDagi35_",
    port="5432",
    sslmode="disable"
)

cur = conn.cursor()

# Insert unique banks
unique_banks = df['bank_name'].unique()

for bank in unique_banks:

    cur.execute("""
        INSERT INTO banks (bank_name, app_name)
        VALUES (%s, %s)
        ON CONFLICT (bank_name) DO NOTHING
    """, (bank, bank))

conn.commit()

# Insert reviews
for _, row in df.iterrows():

    # Get bank_id
    cur.execute("""
        SELECT bank_id
        FROM banks
        WHERE bank_name = %s
    """, (row['bank_name'],))

    bank_id = cur.fetchone()[0]

    # Insert review
    cur.execute("""
        INSERT INTO reviews (
            bank_id,
            review_text,
            rating,
            review_date,
            sentiment_label,
            sentiment_score,
            identified_theme,
            source
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        bank_id,
        row['review_text'],
        int(row['rating']),
        row['review_date'],
        row['sentiment_label'],
        float(row['sentiment_score']),
        row['identified_theme'],
        row['source']
    ))

conn.commit()

print("Data inserted successfully!")

cur.close()
conn.close()

Index(['review text', 'rating (1–5)', 'review date', 'bank / app name',
       'source'],
      dtype='str')
Data inserted successfully!
